In [1]:
import ROOT
import numpy as np
import pandas as pd
 
 # ROOT.gStyle.SetOptFit(0000)    # aggiunge il label con chi2/ndf e probabilita e parametri

#funzione modello
def model(x, par):
    p = x[0]
    A = par[0]
    B = par[1]
    C = par[2]
    return B + (C*(p-A))/((p-A)-C)

def model_derivative(p, par):
    dx=10e-6
    return (model(p + dx, par) - model(p, par)) / dx

#leggere file excel 
xlsx ="esperienza2.xlsx"
df = pd.read_excel("esperienza2.xlsx", sheet_name= "Dati")
df = df.dropna(subset=["p", "q", "sigma_p", "sigma_q"])
df = df.to_numpy(dtype = 'float64')

#definizione variabili
p = df[:, 5]
print(p)
q = df[:, 6]

err_p = df[:, 7]

err_q = df[:,8]

p = np.delete(p, [0,1,2,3,4,5,6,7] )
q = np.delete(q, [0,1,2,3,4,5,6,7] )
err_p = np.delete(err_p, [0,1,2,3,4,5,6,7] )
err_q = np.delete(err_q, [0,1,2,3,4,5,6,7] )




#minimizzazzione con ROOT
model_fit = ROOT.TF1('model', model, min(p), max(p), 3)
model_fit.SetParameters(0.0, 0.0, 0.0)
model_fit.SetParLimits(0, 0, 50) 
model_fit.SetParLimits(1, 0, 50)
model_fit.SetParNames("A", "B", "C")

gr1 = ROOT.TGraphErrors(len(p), p, q, err_p, err_q)
gr1.SetTitle("Dati sperimentali e fit; p; q")
gr1.SetMarkerStyle(20)
gr1.SetMarkerSize(0.7)                    
gr1.SetMarkerColor(ROOT.kAzure + 2)
gr1.SetLineColor(ROOT.kAzure + 3)        
gr1.SetLineWidth(1)


fit_res = gr1.Fit(model_fit, "SR")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par = model_fit.GetParameters()

errors = model_fit.GetParErrors()

chi2 = fit_res.Chi2()

err_qeff = np.sqrt(err_q**2 + ((model_derivative(p, par)) * err_p)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par[0]} ± {errors[0]}")
print(f"B = {par[1]} ± {errors[1]}")
print(f"C = {par[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("APX")


leg1 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg1.AddEntry(gr1, "Dati", "pe")
leg1.AddEntry(model_fit, "Fit", "l")
leg1.Draw()

c1.Draw()
c1.SaveAs("fit_root.png")

[ 232.5  233.5  234.5  235.5  236.5  237.5  238.5  239.5  240.5  241.5
  242.5  243.5  244.5  245.5  246.5  247.5  248.5  249.5  250.5  251.5
  252.5  253.5  254.5  255.5  256.5  257.5  258.5  259.5  260.5  261.5
  262.5  263.5  264.5  265.5  267.5  269.5  271.5  273.5  275.5  277.5
  280.5  281.5  282.5  283.5  284.5  285.5  286.5  287.5  288.5  289.5
  290.5  291.5  292.5  293.5  294.5  295.5  297.5  299.5  301.5  306.5
  311.5  316.5  321.5  326.5  331.5  336.5  341.5  346.5  351.5  356.5
  361.5  366.5  371.5  376.5  381.5  386.5  391.5  396.5  401.5  406.5
  411.5  416.5  421.5  426.5  431.5  441.5  451.5  461.5  471.5  481.5
  491.5  501.5  511.5  521.5  531.5  541.5  551.5  561.5  571.5  581.5
  601.5  621.5  641.5  661.5  681.5  731.5  781.5  831.5  881.5  981.5
 1081.5 1181.5 1281.5 1381.5 1481.5 1631.5]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =     0.132139
NDf                       = Fit status: 0
Parametri stimati:
A 

Info in <TCanvas::Print>: png file fit_root.png has been created


In [2]:
err_R = 1
err_L = 1
err_x0 = 0.115470053837925

if par[0] < 3*(errors[0] + err_x0 + err_R + err_L):
        print("A è compatibile con 0")
else :
        print("A NON è compatibile con 0")

print("---")

if par[1] < 3*(errors[1] + err_x0  + err_L):
    print("B è compatibile con 0")
else :
    print("B NON è compatibile con 0")
 
 


A è compatibile con 0
---
B è compatibile con 0


In [3]:
q_attesi = np.array([model_fit.Eval(x_val) for x_val in p])
residuals_norm = (q - q_attesi) / err_qeff

#residuals_norm = np.delete(residuals_norm, 0)

print(residuals_norm)

gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(q), max(q), 3)
gaussian_fit.SetParameters(1, 0, 1)
gaussian_fit.SetParNames("const", "mu", "sigma")
gaussian_fit.SetLineColor(ROOT.kRed + 1)
gaussian_fit.SetLineWidth(3)

hist = ROOT.TH1F("hist", "Istogramma dei dati; x; y", 20, min(residuals_norm), max(residuals_norm))
hist.Fill(residuals_norm)
hist.SetLineColor(ROOT.kAzure + 3)
hist.SetLineWidth(2)
hist.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)   
hist.GetXaxis().SetTitle("residuo normalizzato")
hist.GetYaxis().SetTitle("conteggi")
hist.SetTitle("Istogramma residui normalizzati")

r = hist.Fit(gaussian_fit, "S")

fitStatus = r.Status()
print(f"Fit status: {fitStatus}")

parameters = gaussian_fit.GetParameters()
errors = gaussian_fit.GetParErrors()

chi2 = gaussian_fit.GetChisquare()

print("Parametri stimati:")
print(f"const = {parameters[0]} ± {errors[0]}")
print(f"#mu = {parameters[1]} ± {errors[1]}")
print(f"#sigma = {parameters[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")


c2 = ROOT.TCanvas("c2", "Istogramma dei dati", 800, 600)
c2.SetGrid()

hist.Draw()

leg2 = ROOT.TLegend(0.13, 0.80, 0.35, 0.89)
leg2.AddEntry(hist, "Residui", "f")
leg2.AddEntry(gaussian_fit, "Fit gaussiano", "l")
leg2.Draw()


c2.Draw()
c2.SaveAs("hist.png")

[-6.90218793e-03 -1.41072702e-02 -1.78600924e-02  7.45234881e-04
 -1.89460450e-02 -1.28045889e-02 -2.03934042e-02  3.24317503e-03
  6.09223948e-03  3.66039203e-03 -2.09218713e-03  9.71910738e-04
 -8.09506869e-03  2.80936474e-03 -1.04858290e-03  8.46081989e-03
 -1.04329339e-03 -4.33075515e-03 -1.24766055e-02 -1.48063208e-02
 -2.16918258e-03  1.54662591e-03 -6.75937681e-03  1.02917319e-02
 -1.02551413e-04 -1.07402644e-03 -1.43999562e-03  1.56099494e-03
  1.52261453e-03  8.21377717e-03  5.80205862e-03  1.94222899e-02
  2.22189258e-03  1.01356062e-02  7.12176104e-03 -1.88586302e-02
 -2.45510355e-02  1.49149317e-03 -1.30733513e-02 -6.94321844e-03
  8.35245891e-03 -4.56850782e-03 -1.64748652e-03  1.26651741e-02
 -2.25412202e-02  5.96256144e-03 -2.11636324e-03  1.59368766e-02
  7.34758269e-03  1.36159057e-02 -1.00624994e-02 -2.15825380e-03
  6.58606518e-03 -1.63415881e-03 -2.30493194e-03 -1.23089815e-03
 -9.81987407e-04 -1.13795919e-03 -9.97972559e-04 -9.63485404e-04
 -1.16626312e-03 -1.31791

Info in <TCanvas::Print>: png file hist.png has been created


In [4]:
if parameters[0] - 1 < 3*errors[0]:
    print("const è compatibile con 1")
else :
    print("const NON è compatibile con 1")

print("---")

if parameters[1] < 3*errors[1]:
    print("mu è compatibile con 0")
else :
    print("mu NON è compatibile con 0")

print("---")

if parameters[2] - 1 < 3*errors[2]:
    print("sigma è compatibile con 1")
else :
    print("sigma NON è compatibile con 1")

const NON è compatibile con 1
---
mu è compatibile con 0
---
sigma è compatibile con 1


In [5]:
def line(x, par):
    x = x[0]
    m = par[0]
    q = par[1]
    return m * x + q

residuals = q - q_attesi
print(residuals)

line_fit = ROOT.TF1('line', line, min(p), max(p), 2)
line_fit.SetParameters(1, 0)
line_fit.SetParNames("m", "q")
line_fit.SetLineColor(ROOT.kRed + 1)
line_fit.SetLineWidth(2)

gr2 = ROOT.TGraphErrors(len(p), p, residuals, np.zeros_like(residuals), err_qeff)
gr2.SetTitle("Residui in funzione di p; p; q - q_{att}")
gr2.SetMarkerStyle(20)
gr2.SetMarkerSize(0.7)
gr2.SetMarkerColor(ROOT.kAzure + 2)
gr2.SetLineColor(ROOT.kAzure + 3)

fit_res = gr2.Fit(line_fit, "S")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = line_fit.GetParameters()
errors = line_fit.GetParErrors()

chi2 = fit_res.Chi2()

print("Parametri stimati:")
print(f"m = {parameters[0]} ± {errors[0]}")
print(f"q = {parameters[1]} ± {errors[1]}")
print(f"Chi2 = {chi2}")

c3 = ROOT.TCanvas("c3", "Fit con ROOT", 800, 600)
c3.SetGrid()

gr2.Draw("APX")

leg3 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg3.AddEntry(gr2, "Residui", "pe")
leg3.AddEntry(line_fit, "Fit lineare", "l")
leg3.Draw()

c3.Draw()
c3.SaveAs("fit_line.png")

[ -5.76913482 -12.63368704 -12.49094766   0.49577613 -13.41202923
  -7.7539291  -12.34940154   2.01925461   3.11764066   2.09170407
  -1.15987642   0.43104992  -3.31405431   1.19805285  -0.56343154
   3.82460125  -0.41821746  -1.77297542  -4.04369038  -5.43017042
  -0.68453505   0.54083774  -2.07542487   2.89667831  -0.0288638
  -0.375575    -0.34388805   0.35947041   0.40257753   1.96154749
   1.53405763   3.8100105    0.60641635   2.85273567   1.94372689
  -3.05605864  -4.81610064   0.2925816   -2.23005131  -0.88828043
   1.28228571  -0.66240143  -0.23887393   2.0524033   -1.73028742
   0.81367474  -0.21660554   1.63110741   1.00268022   1.5096906
  -1.37316851  -0.2577084    0.50555319 -12.34882991 -17.12277258
  -9.13354134  -7.23631543  -8.30803433  -7.2434587   -6.95205711
  -8.35552334  -9.38578022  -7.98336215  -6.0960952   -7.67801271
  -8.68845882 -10.09134305 -10.85451694 -13.94925015 -15.34978796
  -7.03297574  -3.9779389   -2.16580902  -2.57948847  -1.20344747
  -1.0235483

Info in <TCanvas::Print>: png file fit_line.png has been created
